# Import packages

In [1]:
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path
import fitz
import json

# CBO Budget Reports: Text Extraction & Paragraph Stitching Pipeline

## Project Overview
This notebook contains a text-processing pipleline designed to process Congressional Budget Office (CBO) report PDFs spanning from **2000 to 2026**. 

The goal of this script is to parse raw text block structures, strip out repeating headers/footers, and reconstruct paragraphs that were broken up by page layouts or line breaks. The final structured text is exported as a dataframe for text analysis.

---

## Pipeline Mechanics

### 1. Structural Text Extraction
Using the `PyMuPDF` library (`fitz`), text is pulled from the PDF documents using **block layout detection** rather than raw line-by-line reading. This helps preserve native document groupings. During this stage:
* Recurring document artifacts (e.g., *"The Budget and Economic Outlook"*, chapter labels) are removed using case-insensitive regular expressions.
* Hardcoded page layout artifacts and trailing page numbers are filtered out.
* Carriage returns (`\r\n`) are standardized to clean system line breaks (`\n`).

### 2. Context-Aware Paragraph Stitching
Because PDF text extraction frequently splits sentences across artificial page and line boundaries, the pipeline handles reconstruction:
* **Boundary Detection:** If an extracted text block does not end with terminal sentence punctuation (`.`, `!`, `?`, `"`, `”`) or ends directly in a hyphen (`-`), the pipeline recognizes it as an unfinished sentence.
* **Hyphen Resolution:** Trailing hyphens are stripped, and the subsequent text block is cleanly attached back onto the word.
* **Continuous Flow:** Standard words are joined with a space, maintaining an unbroken paragraph narrative.

### 3. Data Schema & Export
The dataset is assigned a normalized structure and tracking metrics:
* **`report_name`**: Target report name.
* **`paragraph_number`**: A sequential paragraph counter that resets for every unique report.
* **`text`**: The cleaned textual block.
* **Empty Classification Targets**: Placeholders (`component`, `category`, `subcategory`) are initialized to prepare the dataframe for future labeling.

The resulting dataset is bundled into a `pandas` DataFrame and automatically exported as a consolidated file to: `data_files/chunked_paragraphs.csv`.

In [11]:
# ======================================================================
# 1. Core Text Extraction & Paragraph Stitching Engine
# ======================================================================

def process_pdf_by_paragraph(pdf_path, pdf_filename):
    """
    Extracts text layout blocks from a CBO PDF, filters out recurring document header/footer
    artifacts, and programmatically stitches sentence fragments together.
    
    Parameters:
        pdf_path (Path): Full filesystem path to the target PDF.
        pdf_filename (str): Base name of the file for dataset tracking.
        
    Returns:
        list: A collection of structured dictionaries containing cleaned paragraphs.
    """
    try:
        doc = fitz.open(pdf_path)
    except Exception as e:
        print(f"❌ Failed to open {pdf_filename}: {e}")
        return []

    raw_chunks = []

    # --- Step 1: Structural Layout Text Extraction ---
    for page in doc:
        # Extract using 'blocks' layout to capture native geometric grouping boundaries
        blocks = page.get_text("blocks")

        for b in blocks:
            text_block = b[4].strip()  # Index 4 contains the raw string contents

            # Remove generic publication titles and section/chapter headers
            text_block = re.sub(
                r"(The Budget and Economic Outlook:|Economic Projections:|Chapter \d+)",
                "",
                text_block,
                flags=re.IGNORECASE,
            )
            # Eliminate notebook page split markers and trailing numeric page digits
            text_block = re.sub(
                r"(---\\s*PAGE\\s*\\d+\\s*---|\\b\\d+\\b\\s*$)", "", text_block
            )

            # Standardize cross-platform carriage returns safely
            text_block = text_block.replace("\r\n", "\n").replace("\r", "\n")

            if text_block.strip():
                raw_chunks.append(text_block.strip())

    doc.close()

    # --- Step 2: Context-Aware Sentence Stitching Pipeline ---
    stitched_paragraphs = []
    buffer_text = ""

    for chunk in raw_chunks:
        # Collapse multi-line interior spaces and breaks down into single spaces
        cleaned_chunk = re.sub(r"\s+", " ", chunk).strip()

        if not cleaned_chunk:
            continue

        if buffer_text:
            last_char = buffer_text[-1]
            # Identify if the previous block was abruptly cut off mid-sentence
            is_cutoff = last_char not in [".", "!", "?", '"', "”"] or buffer_text.endswith("-")

            if is_cutoff:
                # Handle word-hyphenations split across a physical page boundary
                if buffer_text.endswith("-"):
                    buffer_text = buffer_text[:-1] + cleaned_chunk
                else:
                    buffer_text += " " + cleaned_chunk
                continue
            else:
                # Buffer contains a completely closed structural paragraph; save it
                stitched_paragraphs.append(buffer_text)

        buffer_text = cleaned_chunk

    # Flush any remaining text isolated in the final loop buffer
    if buffer_text:
        stitched_paragraphs.append(buffer_text)

    # --- Step 3: Tabular Schema Normalization ---
    paragraph_rows = []
    paragraph_counter = 1

    for p_text in stitched_paragraphs:
        final_clean = re.sub(r"\s+", " ", p_text).strip()

        if final_clean:
            paragraph_rows.append(
                {
                    "report_name": pdf_filename,
                    "paragraph_number": paragraph_counter,
                    "text": final_clean,
                    "component": pd.NA,
                    "category": pd.NA,
                    "subcategory": pd.NA,
                }
            )
            paragraph_counter += 1

    return paragraph_rows


# ======================================================================
# 2. Batch Execution & File Filtering Pipeline (Years 2000-2026)
# ======================================================================

# Establish strict target directories mirroring your repo layout
project_dir = Path.cwd()
reports_dir = project_dir / "cbo_budget_reports"

if not reports_dir.exists() or not reports_dir.is_dir():
    raise ValueError(
        f"The folder directory '{reports_dir}' does not exist. Please check your workspace paths."
    )

# Strict regex to pull a valid 4-digit year format anywhere within the PDF filename
year_pattern = re.compile(r"(20[0-2][0-9])")

# Build a cleanly sorted target file matrix array
pdf_files = [
    p for p in reports_dir.iterdir() if p.is_file() and p.suffix.lower() == ".pdf"
]
pdf_files.sort()

all_pdf_rows = []
print(
    f"Found {len(pdf_files)} total report(s) inside '{reports_dir.name}'. Filtering for 2000-2026 window...\n"
    + "-" * 70
)

# Execute loop across all discovered documents
for file_path in pdf_files:
    pdf_filename = file_path.name
    match = year_pattern.search(pdf_filename)

    if match:
        try:
            report_rows = process_pdf_by_paragraph(file_path, pdf_filename)
            all_pdf_rows.extend(report_rows)
            print(f"✅ Successfully parsed data chunks: {pdf_filename}")
        except Exception as e:
            print(f"❌ Error extracting text from {pdf_filename}: {e}")
    else:
        print(f"⏭️ Skipping (Outside target analytical window): {pdf_filename}")

# Generate consolidated tabular DataFrame matrix
df2 = pd.DataFrame(all_pdf_rows)

# Guarantee the save destination path exists locally prior to writing 
dir_path = Path("data_files")
dir_path.mkdir(parents=True, exist_ok=True)

# Write output out to flat storage
df2.to_csv('data_files/chunked_paragraphs.csv', index=False)
print(f"\n💾 Execution finished. Consolidated data table shape {df2.shape} written to 'data_files/chunked_paragraphs.csv'")

Found 70 total report(s) inside 'cbo_budget_reports'. Filtering for 2000-2026 window...
----------------------------------------------------------------------
✅ Successfully parsed data chunks: 2000-01-01__12069__The Budget and Economic Outlook_ Fiscal Years 2001-2010 (Table 1-6 corrected 2_1_00).pdf
✅ Successfully parsed data chunks: 2000-07-01__12477__The Budget and Economic Outlook_ An Update.pdf
✅ Successfully parsed data chunks: 2001-01-01__12958__The Budget and Economic Outlook_ Fiscal Years 2002-2011.pdf
✅ Successfully parsed data chunks: 2001-08-01__13249__The Budget and Economic Outlook_ An Update.pdf
✅ Successfully parsed data chunks: 2002-01-01__13504__The Budget and Economic Outlook_ Fiscal Years 2003-2012.pdf
✅ Successfully parsed data chunks: 2002-08-27__13956__The Budget and Economic Outlook_ An Update.pdf
✅ Successfully parsed data chunks: 2003-01-01__14254__The Budget and Economic Outlook_ Fiscal Years 2004-2013.pdf
✅ Successfully parsed data chunks: 2003-08-01__14691_